In [0]:
from pyspark.sql import functions as F


class RawDimensionLoader:

    def __init__(
        self,
        source_path: str,
        target_table: str
    ):
        self.source_path = source_path
        self.target_table = target_table


    def read_csv(self):

        return (
            spark.read
                .format("csv")
                .option("header", "true")
                .option("inferSchema", "true")
                .option("mode", "PERMISSIVE")
                .load(self.source_path)
        )


    def add_metadata(self, df):

        return (
            df
            .withColumn(
                "_source_file",
                F.col("_metadata.file_name")
            )
            .withColumn(
                "_load_timestamp",
                F.current_timestamp()
            )
            .withColumn(
                "_load_date",
                F.current_date()
            )
        )


    def write_table(self, df):

        (
            df.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(self.target_table)
        )


    def run(self):

        print(f"Loading {self.source_path}")
        print(f"Target  {self.target_table}")

        df = self.read_csv()

        df = self.add_metadata(df)

        self.write_table(df)

        print(f"Completed: {self.target_table}")

In [0]:
dimensions = [
    {
        "source": "/Volumes/retail_demo/raw/landing/dimensions/customers.csv",
        "table": "retail_demo.raw.customers"
    },
    {
        "source": "/Volumes/retail_demo/raw/landing/dimensions/products.csv",
        "table": "retail_demo.raw.products"
    },
    {
        "source": "/Volumes/retail_demo/raw/landing/dimensions/stores.csv",
        "table": "retail_demo.raw.stores"
    },
    {
        "source": "/Volumes/retail_demo/raw/landing/dimensions/campaigns.csv",
        "table": "retail_demo.raw.campaigns"
    }
]

In [0]:
for dimension in dimensions:

    loader = RawDimensionLoader(
        source_path=dimension["source"],
        target_table=dimension["table"]
    )

    loader.run()